In [ ]:
import os
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['OMP_NUM_THREADS'] = '1'

In [ ]:
import sys
sys.path.append('../')

In [ ]:
import MeshFEM
import mesh, mesh_energy, py_newton_optimizer, viewer, param_utils
import parametrization, benchmark
import numpy as np

In [ ]:
import continuation_parametrization, flip_avoiding_step_length

In [ ]:
from Benchmark import helper_funcs

# Read Mesh and initialization

In [ ]:
m = helper_funcs.read_mesh('../../models/bird_small.msh.xz')

In [ ]:
uv = mesh_energy.NodalVars(m, 2)

In [ ]:
bdry_uv = helper_funcs.getBDdataOnNormalizedCircle(m)
uv_init = helper_funcs.tutteInitialization(m, bdry_uv)

In [ ]:
uv.setVars(uv_init.ravel())

# Prob Set up

In [ ]:
param = continuation_parametrization.symmetric_dirichlet_param(m, uv)
objectives = [param]

In [ ]:
# Construct parametrization energy and problem
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(uv, objectives)
opt = prob.optimizer()

In [ ]:
# opt.options.factorizer = opt.options.factorizer.CatamariAMD

In [ ]:
prob.initialFeasibleStepLengthComputer = flip_avoiding_step_length.FlipAvoidingStepLength(m.elements())
prob.initialFeasibleStepLengthComputer.backoffFactor = 0.8

# Search-over-degree Method

In [ ]:
DEGREE = 3
param.elementHessianShift = 1e-10

In [ ]:
benchmark.reset()
prob.setVars(uv_init.ravel())
gtol = 50
opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionNever()

benchmark.start_timer_section('continuation_iterations')
while True:
    param.setInterpolatedReference(1.0, prob.getVars())
    if np.linalg.norm(prob.gradient()) < gtol: break
    
    param.setInterpolatedReference(0.0)
    prob.invalidateCachedHessian()
    opt.update_factorizations()
    if DEGREE > 0:
        c = param.computeTaylorCoefficients(opt.hessian_factorization, DEGREE)
        c = np.vstack([prob.getVars(), c])
    
    alpha = 0.25
    with benchmark.ScopedTimer('backtracking'):
        bit = 0
        while True:
            param.setInterpolatedReference(alpha)
            chosen_deg = 0
            if (DEGREE > 0):
                min_energy = np.inf
                for deg in range(DEGREE + 1):
                    xa = c[:deg + 1].T @ [alpha**d for d in range(deg + 1)]
                    prob.setVars(xa)
                    e = prob.energy()
                    if e < min_energy:
                        min_energy = e
                        min_energy_x = xa
                        chosen_deg = deg
                prob.setVars(min_energy_x)
            gnorm = np.linalg.norm(prob.gradient())
            if (not np.isinf(prob.energy())) and (gnorm < gtol): break
            print(f'backtracking: {prob.energy(), gnorm}')
            bit += 1
            if (bit == 100): raise Exception('Excessive backtracking')
            alpha *= 0.8
    print(f'Interpolation step size {alpha:0.2} with degree {chosen_deg}')
    
    opt.options.niter = 1
    opt.options.gradTol = gtol if chosen_deg > 0 else 1e-8 # Only force a Newton iteration if degree 0 interpolation is used.
    opt.optimize()
    # print('hessianWasProjected: ', prob.hessianWasProjected)
print('Final optimization')
opt.options.gradTol = 1
opt.options.niter = 300
# opt.options.hessianProjectionController = py_newton_optimizer.MaskedHessianProjectionControllerGradNorm(nf)
# opt.options.hessianProjectionController.verbose = True
# opt.options.hessianProjectionController.percentileControl = True
opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways()
opt.optimize()
benchmark.stop_timer_section('continuation_iterations')
benchmark.report()

# Fixed-degree Method

In [ ]:
DEGREE=3

In [ ]:
benchmark.reset()
prob.setVars(uv_init.ravel())
gtol = 10
opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionNever() # The continuation iterations always use a PSD zero-distortion Hessian (AKVF-like)

benchmark.start_timer_section('continuation_iterations')
while True:
    param.setInterpolatedReference(1.0, prob.getVars())
    if np.linalg.norm(prob.gradient()) < gtol: break
    
    param.setInterpolatedReference(0.0)
    prob.invalidateCachedHessian()
    opt.update_factorizations()
    if DEGREE > 0:
        c = param.computeTaylorCoefficients(opt.hessian_factorization, DEGREE)
        c = np.vstack([prob.getVars(), c])
    
    alpha = 1.0
    with benchmark.ScopedTimer('backtracking'):
        bit = 0
        while True:
            # TODO: fast backtracking
            # Backtracking is almost always due to element flips, especially at higher DEGREE
            # It can therefore speed things up substantially to determine a flip-avoiding
            # step size; this involves finding the smallest root of a `DEGREE * 2` polynomial
            # per element, which likely can be done much more efficiently than
            # the brute-force approach here. (Note that the flip check can be done
            # without any rest config updates via `setInterpolatedReference`).
            param.setInterpolatedReference(alpha)
            if DEGREE > 0:
                with benchmark.ScopedTimer('eval xa'): # This is somehow faster than numpy matmul...
                    xa = c[0].copy()
                    for d in range(1, DEGREE + 1):
                        xa += c[d] * alpha**d
                prob.setVars(xa)
            if np.isinf(prob.energy()):
                print(f'backtracking: inf energy')
            else:
                gnorm = np.linalg.norm(prob.gradient())
                if (gnorm < gtol): break
                print(f'backtracking: {prob.energy(), gnorm}')
            bit += 1
            if (bit == 100): raise Exception('Excessive backtracking')
            alpha *= 0.8
    print(f'Interpolation step size {alpha:0.2}, gnorm: {gnorm:0.3}')
    
    if (DEGREE == 0):
        opt.options.niter = 1
        opt.options.gradTol = 1e-8 # Only force a Newton iteration if degree 0 interpolation is used.
        opt.optimize()
    # print('hessianWasProjected: ', prob.hessianWasProjected)
print('Final optimization')
opt.options.gradTol = 1
opt.options.niter = 300
opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways()
opt.optimize()
benchmark.stop_timer_section('continuation_iterations')
benchmark.report()

# Compare to stretch initialization

In [ ]:
opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAdaptive()
opt.options.hessianProjectionController.startWithProjectionActive = False
opt.options.hessianProjectionController.numProjectionStepsBeforeDisable = 1
opt.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0

In [ ]:
sys.path.append('../Stretch2Relax')
import opt_utils, initial_utils, extra_utils

In [ ]:
uv.setVars(uv_init.ravel())
scale = initial_utils.initialization_scale(m, uv, param, 'grad_minimal')
uv.setVars(scale * uv.getVars())

In [ ]:
param.elementHessianShift = 1e-10

In [ ]:
benchmark.reset()
opt.options.hessianProjectionController.reset()
opt.optimize()
benchmark.report()